In [2]:
%pip install scikit-learn


import pandas as pd
import numpy as np
from itertools import combinations
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# =========================
# LOAD DATA
# =========================

PRICE_FILES = [
    "prices_round_5_day_2.csv",
    "prices_round_5_day_3.csv",
    "prices_round_5_day_4.csv",
]

TRADE_FILES = [
    "trades_round_5_day_2.csv",
    "trades_round_5_day_3.csv",
    "trades_round_5_day_4.csv",
]

prices = pd.concat([pd.read_csv(f, sep=";") for f in PRICE_FILES], ignore_index=True)
trades = pd.concat([pd.read_csv(f, sep=";") for f in TRADE_FILES], ignore_index=True)

prices = prices.rename(columns={"product": "symbol"})
prices["global_t"] = prices["day"] * 1_000_000 + prices["timestamp"]

products = sorted(prices["symbol"].unique())

print("Loaded products:", len(products))
print(products[:10])

# =========================
# BASIC FEATURES
# =========================

def safe_div(a, b):
    return np.where(np.abs(b) > 1e-9, a / b, np.nan)

prices["spread"] = prices["ask_price_1"] - prices["bid_price_1"]
prices["microprice"] = (
    prices["ask_price_1"] * prices["bid_volume_1"] +
    prices["bid_price_1"] * prices["ask_volume_1"]
) / (prices["bid_volume_1"] + prices["ask_volume_1"])

prices["imbalance"] = (
    prices["bid_volume_1"] - prices["ask_volume_1"]
) / (prices["bid_volume_1"] + prices["ask_volume_1"])

prices = prices.sort_values(["symbol", "global_t"])

for h in [1, 2, 5, 10, 25, 50, 100]:
    prices[f"fwd_ret_{h}"] = prices.groupby("symbol")["mid_price"].shift(-h) - prices["mid_price"]

prices["ret_1"] = prices.groupby("symbol")["mid_price"].diff()

# wide mid matrix
mid = prices.pivot_table(index="global_t", columns="symbol", values="mid_price").sort_index()
spread = prices.pivot_table(index="global_t", columns="symbol", values="spread").sort_index()
imb = prices.pivot_table(index="global_t", columns="symbol", values="imbalance").sort_index()

# =========================
# 1. SINGLE PRODUCT MEAN REVERSION / TREND
# =========================

single_rows = []

for sym in products:
    df = prices[prices["symbol"] == sym].copy().dropna(subset=["mid_price"])
    
    for lookback in [20, 50, 100, 200]:
        roll_mean = df["mid_price"].rolling(lookback).mean()
        roll_std = df["mid_price"].rolling(lookback).std()
        z = (df["mid_price"] - roll_mean) / roll_std
        
        for h in [5, 10, 25, 50]:
            tmp = pd.DataFrame({
                "z": z,
                "fwd": df[f"fwd_ret_{h}"]
            }).dropna()
            
            if len(tmp) < 200:
                continue
            
            # mean reversion: high z should predict negative future return
            corr = tmp["z"].corr(tmp["fwd"])
            
            extreme = tmp[np.abs(tmp["z"]) >= 1.5]
            if len(extreme) > 20:
                mr_pnl = np.mean(-np.sign(extreme["z"]) * extreme["fwd"])
                trend_pnl = np.mean(np.sign(extreme["z"]) * extreme["fwd"])
                hit_mr = np.mean((-np.sign(extreme["z"]) * extreme["fwd"]) > 0)
            else:
                mr_pnl = np.nan
                trend_pnl = np.nan
                hit_mr = np.nan
            
            single_rows.append({
                "symbol": sym,
                "lookback": lookback,
                "horizon": h,
                "z_fwd_corr": corr,
                "meanrev_edge": mr_pnl,
                "trend_edge": trend_pnl,
                "meanrev_hit_rate": hit_mr,
                "n_extreme": len(extreme),
                "avg_spread": df["spread"].mean(),
                "vol": df["ret_1"].std(),
            })

single = pd.DataFrame(single_rows)

print("\n=== Best single-product mean reversion candidates ===")
display(
    single.sort_values("meanrev_edge", ascending=False)
    .head(25)
)

print("\n=== Best single-product trend/function continuation candidates ===")
display(
    single.sort_values("trend_edge", ascending=False)
    .head(25)
)

# =========================
# 2. MARKET MAKING QUALITY
# =========================

mm_rows = []

for sym in products:
    df = prices[prices["symbol"] == sym].copy()
    
    avg_spread = df["spread"].mean()
    med_spread = df["spread"].median()
    vol_1 = df["ret_1"].std()
    
    # adverse selection proxy:
    # if imbalance predicts future move, passive MM is dangerous unless adjusted
    imb_corrs = {}
    for h in [1, 5, 10]:
        imb_corrs[f"imb_corr_{h}"] = df["imbalance"].corr(df[f"fwd_ret_{h}"])
    
    # rough MM attractiveness: wide spread, low volatility, low adverse selection
    adv = np.nanmean([abs(v) for v in imb_corrs.values()])
    score = avg_spread / (vol_1 + 1e-9) - 10 * adv
    
    mm_rows.append({
        "symbol": sym,
        "avg_spread": avg_spread,
        "median_spread": med_spread,
        "vol_1": vol_1,
        "adverse_selection_proxy": adv,
        "mm_score": score,
        **imb_corrs,
    })

mm = pd.DataFrame(mm_rows)

print("\n=== Best market-making candidates ===")
display(mm.sort_values("mm_score", ascending=False).head(25))

# =========================
# 3. PAIR STAT ARB / COINTEGRATION-LIKE SPREADS
# =========================

pair_rows = []

for a, b in combinations(products, 2):
    x = mid[a]
    y = mid[b]
    df = pd.concat([x, y], axis=1).dropna()
    df.columns = ["a", "b"]
    
    if len(df) < 500:
        continue
    
    X = df[["b"]].values
    Y = df["a"].values
    
    model = LinearRegression().fit(X, Y)
    beta = model.coef_[0]
    alpha = model.intercept_
    pred = model.predict(X)
    
    resid = df["a"] - pred
    r2 = r2_score(Y, pred)
    
    # half-life proxy
    r = resid.dropna()
    dr = r.diff().dropna()
    lag = r.shift(1).dropna()
    common = pd.concat([dr, lag], axis=1).dropna()
    common.columns = ["dr", "lag"]
    
    if len(common) < 100:
        continue
    
    phi_model = LinearRegression().fit(common[["lag"]], common["dr"])
    phi = phi_model.coef_[0]
    half_life = np.nan
    if phi < 0:
        half_life = -np.log(2) / np.log(1 + phi) if (1 + phi) > 0 else np.nan
    
    z = (resid - resid.rolling(100).mean()) / resid.rolling(100).std()
    
    fwd_resid = resid.shift(-20) - resid
    tmp = pd.DataFrame({"z": z, "fwd_resid": fwd_resid}).dropna()
    extreme = tmp[np.abs(tmp["z"]) >= 1.5]
    
    if len(extreme) > 20:
        pair_edge = np.mean(-np.sign(extreme["z"]) * extreme["fwd_resid"])
        pair_hit = np.mean((-np.sign(extreme["z"]) * extreme["fwd_resid"]) > 0)
    else:
        pair_edge = np.nan
        pair_hit = np.nan
    
    pair_rows.append({
        "a": a,
        "b": b,
        "alpha": alpha,
        "beta": beta,
        "r2": r2,
        "resid_std": resid.std(),
        "half_life": half_life,
        "pair_edge_20": pair_edge,
        "pair_hit_20": pair_hit,
        "n_extreme": len(extreme),
    })

pairs = pd.DataFrame(pair_rows)

print("\n=== Best pair/stat-arb candidates ===")
display(
    pairs[
        (pairs["r2"] > 0.5) &
        (pairs["half_life"] > 1) &
        (pairs["half_life"] < 200)
    ]
    .sort_values("pair_edge_20", ascending=False)
    .head(30)
)

# =========================
# 4. GROUP RELATIONSHIPS / BASKET REVERSION
# =========================

GROUPS = {
    "GALAXY": [p for p in products if p.startswith("GALAXY")],
    "SLEEP_POD": [p for p in products if p.startswith("SLEEP_POD")],
    "MICROCHIP": [p for p in products if p.startswith("MICROCHIP")],
    "PEBBLES": [p for p in products if p.startswith("PEBBLES")],
    "ROBOT": [p for p in products if p.startswith("ROBOT")],
    "UV_VISOR": [p for p in products if p.startswith("UV_VISOR")],
    "TRANSLATOR": [p for p in products if p.startswith("TRANSLATOR")],
    "PANEL": [p for p in products if p.startswith("PANEL")],
    "OXYGEN_SHAKE": [p for p in products if p.startswith("OXYGEN_SHAKE")],
    "SNACKPACK": [p for p in products if p.startswith("SNACKPACK")],
}

basket_rows = []

for group, syms in GROUPS.items():
    if len(syms) < 2:
        continue
    
    W = mid[syms].dropna()
    basket = W.mean(axis=1)
    
    for sym in syms:
        rel = W[sym] - basket
        z = (rel - rel.rolling(100).mean()) / rel.rolling(100).std()
        fwd = W[sym].shift(-20) - W[sym]
        
        tmp = pd.DataFrame({"z": z, "fwd": fwd}).dropna()
        extreme = tmp[np.abs(tmp["z"]) >= 1.5]
        
        if len(extreme) > 20:
            edge = np.mean(-np.sign(extreme["z"]) * extreme["fwd"])
            hit = np.mean((-np.sign(extreme["z"]) * extreme["fwd"]) > 0)
        else:
            edge = np.nan
            hit = np.nan
        
        basket_rows.append({
            "group": group,
            "symbol": sym,
            "basket_reversion_edge_20": edge,
            "basket_hit_20": hit,
            "n_extreme": len(extreme),
        })

basket = pd.DataFrame(basket_rows)

print("\n=== Best within-group basket reversion candidates ===")
display(basket.sort_values("basket_reversion_edge_20", ascending=False).head(30))

# =========================
# 5. CARRY / DRIFT BY PRODUCT
# =========================

carry_rows = []

for sym in products:
    s = mid[sym].dropna()
    
    by_day = prices[prices["symbol"] == sym].groupby("day")["mid_price"]
    day_open = by_day.first()
    day_close = by_day.last()
    day_drift = day_close - day_open
    
    carry_rows.append({
        "symbol": sym,
        "avg_day_drift": day_drift.mean(),
        "day_drift_std": day_drift.std(),
        "consistent_up_days": (day_drift > 0).sum(),
        "consistent_down_days": (day_drift < 0).sum(),
        "total_drift": s.iloc[-1] - s.iloc[0],
        "vol": s.diff().std(),
        "carry_score": abs(day_drift.mean()) / (day_drift.std() + 1e-9),
        "direction": "LONG" if day_drift.mean() > 0 else "SHORT",
    })

carry = pd.DataFrame(carry_rows)

print("\n=== Best carry/drift candidates ===")
display(carry.sort_values("carry_score", ascending=False).head(25))

# =========================
# 6. TRADE FLOW / RECYCLING SIGNALS
# =========================

trade_rows = []

trades = trades.rename(columns={"symbol": "symbol"})
trades["global_t"] = 0

# infer day from file is not present in trades, so reload with day attached
tmp_trades = []
for f in TRADE_FILES:
    day = int(f.split("_day_")[1].split(".")[0])
    t = pd.read_csv(f, sep=";")
    t["day"] = day
    t["global_t"] = day * 1_000_000 + t["timestamp"]
    tmp_trades.append(t)

trades = pd.concat(tmp_trades, ignore_index=True)
trades = trades.rename(columns={"symbol": "symbol"})

# attach future mid moves after public trades
for sym in products:
    tdf = trades[trades["symbol"] == sym].copy()
    pdf = prices[prices["symbol"] == sym][["global_t", "mid_price"]].sort_values("global_t")
    
    if len(tdf) < 20:
        continue
    
    merged = pd.merge_asof(
        tdf.sort_values("global_t"),
        pdf,
        on="global_t",
        direction="backward"
    )
    
    for h in [5, 20, 50]:
        future = pdf.copy()
        future[f"future_mid_{h}"] = future["mid_price"].shift(-h)
        future = future[["global_t", f"future_mid_{h}"]]
        
        merged2 = pd.merge_asof(
            merged.sort_values("global_t"),
            future,
            on="global_t",
            direction="backward"
        )
        
        merged2["fwd"] = merged2[f"future_mid_{h}"] - merged2["mid_price"]
        
        # signed flow unavailable if buyer/seller hidden,
        # but large prints can still predict continuation/reversion
        merged2["notional"] = merged2["price"] * merged2["quantity"]
        
        corr_qty = merged2["quantity"].corr(merged2["fwd"])
        large = merged2[merged2["quantity"] >= merged2["quantity"].quantile(0.75)]
        
        trade_rows.append({
            "symbol": sym,
            "horizon": h,
            "n_trades": len(merged2),
            "qty_fwd_corr": corr_qty,
            "large_trade_avg_fwd": large["fwd"].mean(),
            "large_trade_abs_edge": abs(large["fwd"].mean()),
        })

flow = pd.DataFrame(trade_rows)

print("\n=== Best trade-flow / recycling candidates ===")
display(flow.sort_values("large_trade_abs_edge", ascending=False).head(30))

# =========================
# 7. FINAL COMBINED RANKING
# =========================

final = pd.DataFrame({"symbol": products})

best_mr = single.groupby("symbol")["meanrev_edge"].max()
best_trend = single.groupby("symbol")["trend_edge"].max()
best_mm = mm.set_index("symbol")["mm_score"]
best_basket = basket.groupby("symbol")["basket_reversion_edge_20"].max()
best_carry = carry.set_index("symbol")["carry_score"]
best_flow = flow.groupby("symbol")["large_trade_abs_edge"].max()

final["best_meanrev"] = final["symbol"].map(best_mr)
final["best_trend"] = final["symbol"].map(best_trend)
final["mm_score"] = final["symbol"].map(best_mm)
final["basket_reversion"] = final["symbol"].map(best_basket)
final["carry_score"] = final["symbol"].map(best_carry)
final["flow_edge"] = final["symbol"].map(best_flow)

# normalize columns for rough combined score
score_cols = ["best_meanrev", "best_trend", "mm_score", "basket_reversion", "carry_score", "flow_edge"]

for c in score_cols:
    x = final[c].replace([np.inf, -np.inf], np.nan)
    final[c + "_rankscore"] = x.rank(pct=True)

final["combined_score"] = final[[c + "_rankscore" for c in score_cols]].mean(axis=1)

print("\n=== FINAL PRODUCT RANKING ===")
display(final.sort_values("combined_score", ascending=False).head(50))

print("\nUse this as follows:")
print("1. Pick products that rank high in one clear category, not just combined_score.")
print("2. For market making, prefer high mm_score and low adverse_selection_proxy.")
print("3. For stat arb, use the pair table: trade residual z-score back to zero.")
print("4. For basket reversion, buy product cheap vs group basket and sell rich product.")
print("5. For carry, only trust if direction is consistent across days.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 57.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.3/20.3 MB 46.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [scikit-learn] [scikit-learn]
Note: you may need to restart the kernel to use updated packages.
Loaded products: 50
['GALAXY_SOUNDS_BLACK_HOLES', 'GALAXY_SOUNDS_DARK_MATTER', 'GALAXY_SOUNDS_PLANETARY_RINGS', 'GALAXY_SOUNDS_SOLAR_FLAMES', 'GALAXY_SOUNDS_SOLAR_WINDS', 'MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_RECTANGLE', 'MICROCHIP_SQUARE', 'MICROCHIP_TRIANGLE']

=== Best single-product mean reversion candidates ===


,symbol,lookback,horizon,z_fwd_corr,meanrev_edge,trend_edge,meanrev_hit_rate,n_extreme,avg_spread,vol
383,PEBBLES_XL,200,50,-0.105569,31.922158,-31.922158,0.581515,9121,16.630767,30.314637
375,PEBBLES_XL,50,50,-0.060451,19.497974,-19.497974,0.535623,9376,16.630767,30.314637
379,PEBBLES_XL,100,50,-0.080326,18.116567,-18.116567,0.537136,9398,16.630767,30.314637
382,PEBBLES_XL,200,25,-0.090108,17.556299,-17.556299,0.546102,9121,16.630767,30.314637
371,PEBBLES_XL,20,50,-0.031305,13.365391,-13.365391,0.523874,8859,16.630767,30.314637
374,PEBBLES_XL,50,25,-0.060549,13.363230,-13.363230,0.531104,9388,16.630767,30.314637
378,PEBBLES_XL,100,25,-0.074883,10.392648,-10.392648,0.521864,9399,16.630767,30.314637
127,MICROCHIP_RECTANGLE,200,50,-0.042903,8.324209,-8.324209,0.541218,9571,7.885767,13.130087
370,PEBBLES_XL,20,25,-0.026649,8.075392,-8.075392,0.514943,8867,16.630767,30.314637
447,ROBOT_LAUNDRY,200,50,-0.055913,7.383478,-7.383478,0.529966,9878,7.165067,9.822607



=== Best single-product trend/function continuation candidates ===


,symbol,lookback,horizon,z_fwd_corr,meanrev_edge,trend_edge,meanrev_hit_rate,n_extreme,avg_spread,vol
271,PANEL_1X4,200,50,0.106000,-10.659174,10.659174,0.432179,10454,8.376633,9.484077
267,PANEL_1X4,100,50,0.098208,-8.090107,8.090107,0.445796,10027,8.376633,9.484077
151,MICROCHIP_TRIANGLE,50,50,0.046008,-7.537927,7.537927,0.468708,9571,8.635433,14.503819
155,MICROCHIP_TRIANGLE,100,50,0.044859,-7.113600,7.113600,0.470172,9890,8.635433,14.503819
495,SLEEP_POD_COTTON,200,50,0.053737,-6.343925,6.343925,0.457956,10370,10.050200,11.676242
263,PANEL_1X4,50,50,0.069326,-5.853159,5.853159,0.451430,9718,8.376633,9.484077
143,MICROCHIP_SQUARE,200,50,0.010568,-5.456803,5.456803,0.482632,10047,11.718800,20.707015
63,GALAXY_SOUNDS_SOLAR_FLAMES,200,50,0.046672,-4.857360,4.857360,0.476415,10176,14.071533,11.094265
270,PANEL_1X4,200,25,0.071482,-4.433149,4.433149,0.463757,10471,8.376633,9.484077
503,SLEEP_POD_LAMB_WOOL,50,50,0.048456,-4.390403,4.390403,0.462383,9836,9.400100,10.712194



=== Best market-making candidates ===


...


=== Best trade-flow / recycling candidates ===


,symbol,horizon,n_trades,qty_fwd_corr,large_trade_avg_fwd,large_trade_abs_edge
20,MICROCHIP_OVAL,50,569,-0.046652,-18.907514,18.907514
71,PEBBLES_XL,50,644,-0.015881,16.997175,16.997175
74,PEBBLES_XS,50,644,-0.017704,-13.324859,13.324859
146,UV_VISOR_RED,50,733,0.079229,13.162356,13.162356
73,PEBBLES_XS,20,644,-0.033304,-11.542373,11.542373
64,PEBBLES_M,20,644,0.091304,10.480226,10.480226
62,PEBBLES_L,50,644,0.003735,-9.971751,9.971751
29,MICROCHIP_TRIANGLE,50,569,-0.028134,-9.184971,9.184971
137,UV_VISOR_AMBER,50,733,-0.022427,-8.294540,8.294540
140,UV_VISOR_MAGENTA,50,733,0.037023,7.778736,7.778736



=== FINAL PRODUCT RANKING ===


,symbol,best_meanrev,best_trend,mm_score,basket_reversion,carry_score,flow_edge,best_meanrev_rankscore,best_trend_rankscore,mm_score_rankscore,basket_reversion_rankscore,carry_score_rankscore,flow_edge_rankscore,combined_score
24,PEBBLES_XS,4.420597,0.279407,0.580284,4.118386,2.310727,13.324859,0.78,0.28,0.24,0.96,0.96,0.96,0.696667
23,PEBBLES_XL,31.922158,0.432184,0.373050,8.427986,0.655464,16.997175,1.00,0.34,0.04,1.00,0.68,0.98,0.673333
21,PEBBLES_M,4.968263,1.385739,0.730554,4.150088,0.143888,10.480226,0.86,0.56,0.52,0.98,0.18,0.92,0.670000
3,GALAXY_SOUNDS_SOLAR_FLAMES,2.427767,4.857360,0.958662,2.004388,0.271104,3.454023,0.54,0.92,0.86,0.84,0.34,0.50,0.666667
46,UV_VISOR_MAGENTA,4.588922,-0.552203,0.875551,2.516934,0.708835,7.778736,0.80,0.04,0.68,0.88,0.72,0.84,0.660000
6,MICROCHIP_OVAL,1.860087,2.600965,0.499447,0.157351,2.304885,18.907514,0.44,0.76,0.12,0.46,0.94,1.00,0.620000
48,UV_VISOR_RED,0.727782,3.945353,0.899096,-1.840959,1.654021,13.162356,0.22,0.82,0.76,0.08,0.88,0.94,0.616667
36,SNACKPACK_PISTACHIO,2.513967,-0.150828,2.253285,0.449066,1.626788,3.580460,0.58,0.14,1.00,0.58,0.86,0.52,0.613333
45,UV_VISOR_AMBER,0.162959,4.098627,0.932513,-1.314818,1.499671,8.294540,0.14,0.84,0.84,0.12,0.82,0.86,0.603333
35,SNACKPACK_CHOCOLATE,0.204920,2.527307,1.796103,-0.668014,1.928609,3.629310,0.16,0.74,0.96,0.30,0.92,0.54,0.603333



Use this as follows:
1. Pick products that rank high in one clear category, not just combined_score.
2. For market making, prefer high mm_score and low adverse_selection_proxy.
3. For stat arb, use the pair table: trade residual z-score back to zero.
4. For basket reversion, buy product cheap vs group basket and sell rich product.
5. For carry, only trust if direction is consistent across days.
